# classification using pose

In [1]:
import time as t
import os
import numpy as np
from ultralytics import YOLO
from glob import glob
import cv2
import json
import yaml
import tqdm as tqdm
from pathlib import Path
import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tqdm 


from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image, scale_cam_image

# config

In [2]:
dataset_dir = "../datasets/"
dataset_path = Path(dataset_dir)
posecls_dataset_dir = dataset_path / "AllSpecies-posecls"
groups = ["Coleoptera", "Hymenoptera", "Lepidoptera"]
config_file = "yolo26n-pose.pt"

## preparation

In [3]:
all_test_images = [img for img in posecls_dataset_dir.rglob("**/*.png") if "test" in img.parts] + [img for img in posecls_dataset_dir.rglob("**/*.jpg") if "test" in img.parts]
print(f"Total test images: {len(all_test_images)}")

Total test images: 145


 # yolo loading

In [4]:
model = YOLO(config_file)

# training

In [8]:
results = model.train(data="..\\datasets\\AllSpecies-posecls\\yolo-config.yaml", epochs=1, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.59 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.27  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\datasets\AllSpecies-posecls\yolo-config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=

# evaluation

In [ ]:
y_true = []
y_pred = []

for img in tqdm.tqdm(all_test_images):
    true_label = img.parts[-2]  # Assuming the folder name is the class label
    y_true.append(true_label)
    
    result = model(str(img), conf=0.25)[0]

    print(result.boxes.cls)
    if len(result.boxes.cls) > 0:
        pred_label = result.boxes.cls[0].item()  # Get the predicted class index
        y_pred.append(groups[pred_label])  # Map index to class name
    else:
        y_pred.append("No Detection")  # Handle case where no detection is made
    


  0%|          | 0/145 [00:00<?, ?it/s]


image 1/1 c:\Users\tombe\Documents\_MLE\CV-for-GRIT\models\classifier\..\datasets\AllSpecies-posecls\images\test\IMG_0109_specimen_3_MECKON_NEON.BET.D20.000026.png: 640x384 (no detections), 192.4ms
Speed: 2.1ms preprocess, 192.4ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)


  0%|          | 0/145 [00:00<?, ?it/s]

ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: ultralytics.engine.results.Keypoints object
masks: None
names: {0: 'Coleoptera', 1: 'Hymenoptera', 2: 'Lepidoptera'}
obb: None
orig_img: array([[[214, 210, 209],
        [216, 211, 210],
        [215, 210, 209],
        ...,
        [217, 212, 209],
        [217, 212, 209],
        [217, 212, 209]],

       [[214, 210, 209],
        [217, 212, 211],
        [217, 212, 211],
        ...,
        [216, 211, 208],
        [217, 212, 209],
        [217, 212, 209]],

       [[214, 210, 209],
        [215, 210, 209],
        [215, 210, 209],
        ...,
        [215, 212, 208],
        [217, 212, 209],
        [217, 212, 209]],

       ...,

       [[224, 218, 211],
        [223, 217, 212],
        [224, 218, 213],
        ...,
        [215, 211, 210],
        [214, 210, 209],
        [214, 210, 209]],

       [[224, 218, 211],
        [223, 217, 212],
        [225, 219, 214

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot().figure_.savefig('posecls_confusion_matrix.png')
disp.plot()